# Daymet + PCA20, seed 46

Fixed-budget repeatability run. Same basin IDs, years, model architecture, preprocessing and training budget as the verified seed-42 baseline. Only the random seed and result directory change. The CSV beside this notebook is the sole training data source.


## 1. Read files and define the experiment

In [1]:
from pathlib import Path
import csv, json, time, random, hashlib, base64, zlib, math, os
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Sampler
from IPython.display import display, HTML
CSV_PATH = Path.cwd() / 'daymet_pca20_discharge_1980_2009.csv'
RUN_NAME = 'results_seed46'
OUT = CSV_PATH.parent / RUN_NAME
FEATURES = ['prcp', 'tmax', 'tmin', 'srad', 'vp', 'dayl', 'pet']
STATIC_FEATURES = ['PC01_mean', 'PC01_std', 'PC02_mean', 'PC02_std', 'PC03_mean', 'PC03_std', 'PC04_mean', 'PC04_std', 'PC05_mean', 'PC05_std', 'PC06_mean', 'PC06_std', 'PC07_mean', 'PC07_std', 'PC08_mean', 'PC08_std', 'PC09_mean', 'PC09_std', 'PC10_mean', 'PC10_std', 'PC11_mean', 'PC11_std', 'PC12_mean', 'PC12_std', 'PC13_mean', 'PC13_std', 'PC14_mean', 'PC14_std', 'PC15_mean', 'PC15_std', 'PC16_mean', 'PC16_std', 'PC17_mean', 'PC17_std', 'PC18_mean', 'PC18_std', 'PC19_mean', 'PC19_std', 'PC20_mean', 'PC20_std']
CFG = dict(seed=46, epochs=20, batch=128, windows_per_basin=256, validation_cap=128, lookback=365, hidden=64, learning_rate=0.001, gradient_clip=1.0, train_start='1980-01-01', validation_start='2000-01-01', test_start='2005-01-01', end_exclusive='2010-01-01', features=FEATURES, static_features=STATIC_FEATURES)
assert CSV_PATH.is_file(), 'Put the CSV beside this notebook, or edit CSV_PATH.'
assert torch.cuda.is_available(), 'Select a GPU Jupyter session/kernel.'
torch.set_num_threads(2)
random.seed(46)
np.random.seed(46)
torch.manual_seed(46)
torch.cuda.manual_seed_all(46)
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))
print('CSV:', CSV_PATH)
print('Only model inputs:', FEATURES)
print('Static inputs:', STATIC_FEATURES)


GPU: NVIDIA H100 80GB HBM3 MIG 1g.10gb
CSV: /home/pollen26/Desktop/LSTM_RR_Modelling/Daymet_PCA10_PCA20_Five_Seed_Comparison/PCA20/seed_46/daymet_pca20_discharge_1980_2009.csv
Only model inputs: ['prcp', 'tmax', 'tmin', 'srad', 'vp', 'dayl', 'pet']
Static inputs: ['PC01_mean', 'PC01_std', 'PC02_mean', 'PC02_std', 'PC03_mean', 'PC03_std', 'PC04_mean', 'PC04_std', 'PC05_mean', 'PC05_std', 'PC06_mean', 'PC06_std', 'PC07_mean', 'PC07_std', 'PC08_mean', 'PC08_std', 'PC09_mean', 'PC09_std', 'PC10_mean', 'PC10_std', 'PC11_mean', 'PC11_std', 'PC12_mean', 'PC12_std', 'PC13_mean', 'PC13_std', 'PC14_mean', 'PC14_std', 'PC15_mean', 'PC15_std', 'PC16_mean', 'PC16_std', 'PC17_mean', 'PC17_std', 'PC18_mean', 'PC18_std', 'PC19_mean', 'PC19_std', 'PC20_mean', 'PC20_std']


In [2]:
EXPECTED_SPLIT_HASH = '779bde7ce5db72c8158f26f161865c4987eb4ce6a0ac7469383256b41cc2d844'

def atomic(path, obj):
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, indent=2))
    tmp.replace(path)

def save_csv(path, rows, headers=None):
    headers = headers or list(rows[0])
    with open(path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=headers)
        w.writeheader()
        w.writerows(rows)


In [3]:
from itertools import groupby
records = []
preview = []
expected_dates = np.arange(np.datetime64('1980-01-01'), np.datetime64('2010-01-01'))
started = time.time()
with open(CSV_PATH, newline='') as f:
    reader = csv.reader(f)
    header = next(reader)
    expected = ['basin_id', 'split', 'date', *FEATURES, 'q_obs_mm_day', *STATIC_FEATURES, 'seen_unseen', 'n_subbasins', 'single_subbasin_flag', 'time_split']
    assert header == expected, header
    for basin, group in groupby(reader, key=lambda row: row[0]):
        chunk = list(group)
        splits = {r[1] for r in chunk}
        assert len(splits) == 1
        split = next(iter(splits))
        dates = np.array([r[2] for r in chunk], dtype='datetime64[D]')
        assert np.array_equal(dates, expected_dates), basin
        data = np.array([r[3:11] for r in chunk], dtype='float32')
        assert data.shape == (len(dates), 8)
        static = np.array(chunk[0][11:51], dtype='float64')
        assert np.isfinite(static).all()
        assert all((r[11:54] == chunk[0][11:54] for r in chunk)), 'Static feature or basin metadata changes within basin'
        seen = 'Seen' if split == 'training_donor' else 'Unseen'
        assert chunk[0][51] == seen
        expected_period = np.where(dates < np.datetime64('2000-01-01'), 'train', np.where(dates < np.datetime64('2005-01-01'), 'validation', 'test'))
        assert np.array_equal(np.array([r[54] for r in chunk]), expected_period)
        assert np.all(data[np.isfinite(data[:, -1]), -1] >= 0)
        records.append(dict(id=basin, split=split, dates=dates, x=data[:, :7].copy(), y=data[:, 7].copy(), static=static, n_subbasins=int(chunk[0][52]), single_subbasin_flag=int(chunk[0][53])))
        if not preview:
            preview = chunk[:5]
        if len(records) % 200 == 0:
            print('CSV catchments loaded:', len(records), flush=True)
ids = [r['id'] for r in records]
assert len(ids) == len(set(ids)) == 1426
split_text = ''.join((r['id'] + '|' + r['split'] + chr(10) for r in sorted(records, key=lambda r: r['id'])))
assert hashlib.sha256(split_text.encode()).hexdigest() == EXPECTED_SPLIT_HASH, 'Catchment split differs from original baseline.'
donors = [r for r in records if r['split'] == 'training_donor']
held = [r for r in records if r['split'] == 'unseen_test']
assert len(donors) == 1326 and len(held) == 100 and (not {r['id'] for r in donors} & {r['id'] for r in held})
hash_csv = hashlib.sha256()
with open(CSV_PATH, 'rb') as f:
    for part in iter(lambda: f.read(8 * 1024 * 1024), b''):
        hash_csv.update(part)
CSV_HASH = hash_csv.hexdigest()
OUT.mkdir(exist_ok=True)
DIGEST = hashlib.sha256(json.dumps(dict(config=CFG, csv_sha256=CSV_HASH), sort_keys=True).encode()).hexdigest()
atomic(OUT / 'data_audit.json', dict(csv=str(CSV_PATH), rows=sum((len(r['dates']) for r in records)), basins=1426, donors=1326, unseen=100, csv_sha256=CSV_HASH, dynamic_inputs=FEATURES, static_inputs=STATIC_FEATURES, metadata_excluded=['basin_id', 'split', 'seen_unseen', 'n_subbasins', 'single_subbasin_flag', 'time_split'], loader_seconds=round(time.time() - started)))
display(HTML('<table border="1">' + ''.join(('<tr>' + ''.join(('<td>' + str(x) + '</td>' for x in row)) + '</tr>' for row in [header] + preview)) + '</table>'))
print('CSV is the sole data source. Exact original split verified.')


CSV catchments loaded: 200


CSV catchments loaded: 400


CSV catchments loaded: 600


CSV catchments loaded: 800


CSV catchments loaded: 1000


CSV catchments loaded: 1200


CSV catchments loaded: 1400


basin_id,split,date,prcp,tmax,tmin,srad,vp,dayl,pet,q_obs_mm_day,PC01_mean,PC01_std,PC02_mean,PC02_std,PC03_mean,PC03_std,PC04_mean,PC04_std,PC05_mean,PC05_std,PC06_mean,PC06_std,PC07_mean,PC07_std,PC08_mean,PC08_std,PC09_mean,PC09_std,PC10_mean,PC10_std,PC11_mean,PC11_std,PC12_mean,PC12_std,PC13_mean,PC13_std,PC14_mean,PC14_std,PC15_mean,PC15_std,PC16_mean,PC16_std,PC17_mean,PC17_std,PC18_mean,PC18_std,PC19_mean,PC19_std,PC20_mean,PC20_std,seen_unseen,n_subbasins,single_subbasin_flag,time_split
CAN_01AD003,training_donor,1980-01-01,0.630463481,-7.76700449,-15.3141403,128.286835,185.841629,29947.8027,0.0535097606,0.687992156,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,0.09899644862646227,0.2812285695468155,-0.24155867689115967,0.34717060146034256,-0.36674693628286864,0.19063915690040847,0.36610988885224255,0.22719386485410673,0.019025138598094588,0.13801309308430423,-0.060425384662737835,0.21713313557280223,-0.07774287381280943,0.18720687328007155,-0.16504573796172692,0.14852277965396177,0.007552352447023098,0.14935879370919702,-0.0962941612373958,0.11514251641686565,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-02,0,-6.98227119,-16.2555332,154.34343,171.882034,30004.3125,0.0111347819,0.662272871,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,0.09899644862646227,0.2812285695468155,-0.24155867689115967,0.34717060146034256,-0.36674693628286864,0.19063915690040847,0.36610988885224255,0.22719386485410673,0.019025138598094588,0.13801309308430423,-0.060425384662737835,0.21713313557280223,-0.07774287381280943,0.18720687328007155,-0.16504573796172692,0.14852277965396177,0.007552352447023098,0.14935879370919702,-0.0962941612373958,0.11514251641686565,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-03,0,-13.8265314,-20.5408897,122.875526,119.336349,30065.4141,0.058900591,0.655843019,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,0.09899644862646227,0.2812285695468155,-0.24155867689115967,0.34717060146034256,-0.36674693628286864,0.19063915690040847,0.36610988885224255,0.22719386485410673,0.019025138598094588,0.13801309308430423,-0.060425384662737835,0.21713313557280223,-0.07774287381280943,0.18720687328007155,-0.16504573796172692,0.14852277965396177,0.007552352447023098,0.14935879370919702,-0.0962941612373958,0.11514251641686565,Seen,31,0,train
CAN_01AD003,training_donor,1980-01-04,0,-11.6368589,-21.8019562,167.424911,106.858482,30131.0566,0.00925885979,0.635267556,-0.2066387757796508,0.2367385661863031,0.16099716386303808,0.9023929407661494,0.14071203531066492,0.14469282241915343,0.47427492044915764,0.42726704283066036,0.46972795686659036,0.2801243586851001,-0.17954546782990327,0.18870330064673368,-0.11986888614792282,0.25609750341397913,-0.011018393380067819,0.4291014723459691,-0.19360695346111592,0.2846932747858513,-0.15859346090629897,0.48121343013438833,0.09899644862646227,0.2812285695468155,-0.24155867689115967,0.34717060146034256,-0.36674693628286864,0.19063915690040847,0.366109

CSV is the sole data source. Exact original split verified.


## 2. Prepare data using training basins and years only
Fit forcing/flow scalers on donor training years and static scalers once per donor basin. Create valid meteorology windows and equal-basin training sampling.

In [4]:
def fit_preprocessing(rs):
    assert len(rs) == 1326 and all((r['split'] == 'training_donor' for r in rs))
    sx = np.zeros(7)
    sx2 = np.zeros(7)
    nx = 0
    sy = 0.0
    sy2 = 0.0
    ny = 0
    sigmas = []
    for r in rs:
        train = r['dates'] < np.datetime64(CFG['validation_start'])
        x = r['x'][train & np.isfinite(r['x']).all(1)].astype('float64')
        y = r['y'][train & np.isfinite(r['y'])].astype('float64')
        assert len(x) > 0 and len(y) > 0, r['id']
        sx += x.sum(0)
        sx2 += (x * x).sum(0)
        nx += len(x)
        sy += y.sum()
        sy2 += (y * y).sum()
        ny += len(y)
        sigmas.append(float(y.std()))
    p = dict(xmean=sx / nx, xstd=np.maximum(np.sqrt(np.maximum(sx2 / nx - (sx / nx) ** 2, 0)), 1e-06), ymean=sy / ny, ystd=max(np.sqrt(max(sy2 / ny - (sy / ny) ** 2, 0)), 1e-06))
    p['loss_scales'] = np.maximum(np.array(sigmas) / p['ystd'], 0.1)
    return p

class Windows(Dataset):

    def __init__(self, rs, p, split, cap=None):
        self.rs, self.p, self.split = (rs, p, split)
        self.x = []
        self.indices = []
        for r in rs:
            x = ((r['x'] - p['xmean']) / p['xstd']).astype('float32')
            self.x.append(x)
            d = r['dates']
            L = CFG['lookback']
            t = np.arange(L - 1, len(d))
            bad = np.r_[0, np.cumsum(~np.isfinite(x).all(1))]
            keep = bad[t + 1] - bad[t + 1 - L] == 0
            if split == 'train':
                keep &= d[t] < np.datetime64(CFG['validation_start'])
            elif split == 'validation':
                keep &= (d[t] >= np.datetime64(CFG['validation_start'])) & (d[t] < np.datetime64(CFG['test_start']))
            elif split == 'test':
                keep &= d[t] >= np.datetime64(CFG['test_start'])
            else:
                raise ValueError(split)
            if split != 'test':
                keep &= np.isfinite(r['y'][t])
            ix = t[keep]
            if cap and len(ix) > cap:
                ix = ix[np.linspace(0, len(ix) - 1, cap).astype(int)]
            self.indices.append(ix.astype('int32'))
        self.lengths = np.array([len(a) for a in self.indices])
        self.ends = np.cumsum(self.lengths)
        self.starts = np.r_[0, self.ends[:-1]]
        if split == 'train':
            assert (self.lengths > 0).all()

    def __len__(self):
        return int(self.ends[-1])

    def __getitem__(self, i):
        bi = int(np.searchsorted(self.ends, i, side='right'))
        t = int(self.indices[bi][i - self.starts[bi]])
        L = CFG['lookback']
        y = (self.rs[bi]['y'][t] - self.p['ymean']) / self.p['ystd'] if self.split != 'test' else 0.0
        dynamic = self.x[bi][t + 1 - L:t + 1]
        static = np.broadcast_to(self.rs[bi]['static_scaled'], (L, len(STATIC_FEATURES)))
        inputs = np.concatenate([dynamic, static], axis=1)
        return (torch.from_numpy(inputs), torch.tensor(y, dtype=torch.float32), bi, t)

class EqualBasinSampler(Sampler):

    def __init__(self, ds, per_basin=256, seed=46):
        self.ds, self.per_basin, self.seed, self.epoch = (ds, per_basin, seed, 0)

    def __len__(self):
        return len(self.ds.rs) * self.per_basin

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        bs = np.repeat(np.arange(len(self.ds.rs)), self.per_basin)
        rng.shuffle(bs)
        ix = self.ds.starts[bs] + rng.integers(self.ds.lengths[bs])
        return iter(ix.tolist())
p = fit_preprocessing(donors)
static_train = np.stack([r['static'] for r in donors])
p['static_mean'] = static_train.mean(0)
p['static_std'] = np.maximum(static_train.std(0), 1e-06)
assert np.isfinite(p['static_mean']).all() and np.isfinite(p['static_std']).all()
for r in records:
    r['static_scaled'] = ((r['static'] - p['static_mean']) / p['static_std']).astype('float32')
train = Windows(donors, p, 'train')
val = Windows(donors, p, 'validation', cap=128)
sampler = EqualBasinSampler(train)
loader = DataLoader(train, batch_size=128, sampler=sampler, num_workers=0, pin_memory=True)
val_loader = DataLoader(val, batch_size=256, num_workers=0, pin_memory=True)
assert len(sampler) == 1326 * 256
r = held[0]
altered = dict(r, y=np.full_like(r['y'], np.nan))
ta = Windows([r], p, 'test')
tb = Windows([altered], p, 'test')
assert np.array_equal(ta.indices[0], tb.indices[0])
assert torch.equal(ta[0][0], tb[0][0])
assert ta[0][1].item() == 0.0
np.savez(OUT / 'preprocessing.npz', **p)
atomic(OUT / 'fit_metadata.json', dict(fitted_ids=[r['id'] for r in donors], excluded_ids=[r['id'] for r in held], fit_end_exclusive='2000-01-01', static_features=STATIC_FEATURES, experiment_hash=DIGEST))
save_csv(OUT / 'window_counts.csv', [dict(basin_id=r['id'], train_windows=int(n), validation_windows=int(v)) for r, n, v in zip(donors, train.lengths, val.lengths)])
print('Training windows:', len(train), 'fixed validation windows:', len(val), 'validation donors:', int((val.lengths > 0).sum()))
assert train[0][0].shape == (365, 47)
assert not {r['id'] for r in held} & {r['id'] for r in train.rs}
print('Static scaler fitted once per donor basin; 47 LSTM input channels.')


Training windows: 8129370 fixed validation windows: 165015 validation donors: 1290
Static scaler fitted once per donor basin; 47 LSTM input channels.


## 3. Train and evaluate on validation years
Checkpoint after every epoch, resume safely if interrupted, and select the lowest donor validation loss.

In [5]:
class MeteorologyLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(47, 64, batch_first=True)
        self.head = nn.Linear(64, 1)

    def forward(self, x):
        assert x.shape[-1] == 47
        z, _ = self.lstm(x)
        return self.head(z[:, -1]).squeeze(-1)

def checkpoint(path, model, opt, epoch, best, history):
    state = dict(model=model.state_dict(), optimizer=opt.state_dict(), epoch=epoch, best_validation=best, history=history, experiment_hash=DIGEST, python_rng=random.getstate(), numpy_rng=np.random.get_state(), torch_rng=torch.get_rng_state(), cuda_rng=torch.cuda.get_rng_state_all())
    tmp = path.with_suffix('.tmp')
    torch.save(state, tmp)
    tmp.replace(path)

def validate(model):
    model.eval()
    sums = np.zeros(1326)
    nums = np.zeros(1326)
    with torch.no_grad():
        for x, y, bi, t in val_loader:
            q = model(x.to(DEVICE, non_blocking=True)).cpu().numpy()
            b = bi.numpy()
            err = (q - y.numpy()) ** 2 / p['loss_scales'][b] ** 2
            sums += np.bincount(b, weights=err, minlength=1326)
            nums += np.bincount(b, minlength=1326)
    return (float(np.mean(sums[nums > 0] / nums[nums > 0])), int((nums > 0).sum()))
model = MeteorologyLSTM().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=0.001)
assert model.lstm.input_size == 47 and sum((v.numel() for v in model.parameters())) == 28993
x, y, bi, t = next(iter(loader))
q = model(x[:4].to(DEVICE))
scale = torch.tensor(p['loss_scales'][bi[:4].numpy()], dtype=torch.float32, device=DEVICE)
loss = (((q - y[:4].to(DEVICE)) / scale) ** 2).mean()
loss.backward()
assert torch.isfinite(loss)
opt.zero_grad()
torch.cuda.synchronize()
first = 0
best = float('inf')
history = []
if (OUT / 'last.pt').exists():
    ck = torch.load(OUT / 'last.pt', map_location='cpu', weights_only=False)
    assert ck['experiment_hash'] == DIGEST, 'Config/CSV changed; use another RUN_NAME.'
    model.load_state_dict(ck['model'])
    opt.load_state_dict(ck['optimizer'])
    first = ck['epoch']
    best = ck['best_validation']
    history = ck['history']
    random.setstate(ck['python_rng'])
    np.random.set_state(ck['numpy_rng'])
    torch.set_rng_state(ck['torch_rng'].cpu())
    torch.cuda.set_rng_state_all(ck['cuda_rng'])
started = time.time()
for epoch in range(first, 20):
    model.train()
    sampler.epoch = epoch
    total = 0.0
    count = 0
    for step, (x, y, bi, t) in enumerate(loader, 1):
        opt.zero_grad(set_to_none=True)
        q = model(x.to(DEVICE, non_blocking=True))
        target = y.to(DEVICE, non_blocking=True)
        scale = torch.tensor(p['loss_scales'][bi.numpy()], dtype=torch.float32, device=DEVICE)
        loss = (((q - target) / scale) ** 2).mean()
        assert torch.isfinite(loss)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += float(loss.detach()) * len(y)
        count += len(y)
        if step % 400 == 0:
            atomic(OUT / 'status.json', dict(phase='training', epoch=epoch + 1, batch=step, batches=len(loader)))
            print('PROGRESS', epoch + 1, step, flush=True)
    v, nval = validate(model)
    history.append(dict(epoch=epoch + 1, training_loss=total / count, validation_loss=v, validation_donors=nval, elapsed_seconds=round(time.time() - started)))
    improved = v < best
    best = min(best, v)
    checkpoint(OUT / 'last.pt', model, opt, epoch + 1, best, history)
    if improved:
        checkpoint(OUT / 'best.pt', model, opt, epoch + 1, best, history)
    atomic(OUT / 'history.json', history)
    print('EPOCH', history[-1], flush=True)
ck = torch.load(OUT / 'best.pt', map_location='cpu', weights_only=False)
model.load_state_dict(ck['model'])
model.eval()
print('Best epoch:', ck['epoch'])
atomic(OUT / 'run_config.json', dict(config=CFG, csv_sha256=CSV_HASH, model_inputs=FEATURES, static_features=STATIC_FEATURES, parameters=sum((v.numel() for v in model.parameters())), torch_version=torch.__version__, numpy_version=np.__version__, gpu=torch.cuda.get_device_name(0)))


PROGRESS 1 400


PROGRESS 1 800


PROGRESS 1 1200


PROGRESS 1 1600


PROGRESS 1 2000


PROGRESS 1 2400


... (131 outputs left out)

PROGRESS 20 2000


PROGRESS 20 2400


EPOCH {'epoch': 20, 'training_loss': 0.24527264228526105, 'validation_loss': 0.24638407159914616, 'validation_donors': 1290, 'elapsed_seconds': 591}


Best epoch: 19


## 4. Test the frozen best model on seen and unseen catchments
Predict daily flow for 2005–2009 in both groups. Observed flow is used only to score predictions. NSE, original KGE and volume bias (%) use finite observed/predicted pairs; missing/constant cases retain explicit status.

In [6]:
def scores(y, q):
    good = np.isfinite(y) & np.isfinite(q)
    y, q = (y[good], q[good])
    if len(y) < 2 or y.std() == 0:
        return dict(n=len(y), NSE=np.nan, KGE=np.nan, bias_percent=np.nan)
    nse = 1 - np.sum((q - y) ** 2) / np.sum((y - y.mean()) ** 2)
    corr = np.corrcoef(y, q)[0, 1] if q.std() > 0 else np.nan
    kge = 1 - np.sqrt((corr - 1) ** 2 + (q.std() / y.std() - 1) ** 2 + (q.mean() / y.mean() - 1) ** 2) if y.mean() != 0 else np.nan
    return dict(n=len(y), NSE=float(nse), KGE=float(kge), bias_percent=float(100 * (q.sum() - y.sum()) / y.sum()) if y.sum() != 0 else np.nan)

def evaluate(rs, tag):
    folder = OUT / tag
    folder.mkdir(exist_ok=True)
    metrics = []
    for i, r in enumerate(rs):
        ds = Windows([r], p, 'test')
        ts = []
        pred = []
        with torch.no_grad():
            for x, y, bi, t in DataLoader(ds, batch_size=256, num_workers=0, pin_memory=True):
                q = model(x.to(DEVICE, non_blocking=True)).cpu().numpy() * p['ystd'] + p['ymean']
                ts.extend(t.numpy().tolist())
                pred.extend(q.tolist())
        ts = np.array(ts, dtype=int)
        q = np.array(pred)
        obs = r['y'][ts]
        assert np.isfinite(q).all() and np.all(r['dates'][ts] >= np.datetime64('2005-01-01')) and np.all(r['dates'][ts] < np.datetime64('2010-01-01'))
        save_csv(folder / (r['id'] + '_predictions.csv'), [dict(date=str(r['dates'][t]), observed_mm_day=float(o), predicted_mm_day=float(v)) for t, o, v in zip(ts, obs, q)], ['date', 'observed_mm_day', 'predicted_mm_day'])
        good = np.isfinite(obs) & np.isfinite(q)
        error = q[good] - obs[good]
        extra = dict(RMSE=float(np.sqrt(np.mean(error ** 2))) if len(error) else np.nan, MAE=float(np.mean(np.abs(error))) if len(error) else np.nan)
        metrics.append(dict(basin_id=r['id'], **scores(obs, q), **extra))
        if (i + 1) % 100 == 0 or i + 1 == len(rs):
            print('TEST', tag, i + 1, '/', len(rs), flush=True)
            atomic(OUT / 'status.json', dict(phase='testing', test=tag, completed=i + 1, total=len(rs)))
    save_csv(folder / 'metrics.csv', metrics)
    return metrics
unseen_metrics = evaluate(held, 'unseen100')
temporal_metrics = evaluate(donors, 'donor_temporal')
all_metrics = []
for tag, rows in [('Seen', temporal_metrics), ('Unseen', unseen_metrics)]:
    for r in rows:
        r = dict(r, seen_unseen=tag, catchment_split='training_donor' if tag == 'Seen' else 'unseen_test', test_start='2005-01-01', test_end='2009-12-31', model='Daymet + PCA20')
        r['status'] = 'scored' if r['n'] >= 2 and np.isfinite(r['NSE']) else 'no_test_observations' if r['n'] == 0 else 'insufficient_or_constant_observations'
        all_metrics.append(r)
all_metrics.sort(key=lambda r: r['basin_id'])
assert len(all_metrics) == len({r['basin_id'] for r in all_metrics}) == 1426
assert sum((r['seen_unseen'] == 'Seen' for r in all_metrics)) == 1326
assert sum((r['seen_unseen'] == 'Unseen' for r in all_metrics)) == 100
save_csv(CSV_PATH.parent / 'basin_model_performance.csv', all_metrics, ['basin_id', 'seen_unseen', 'catchment_split', 'model', 'test_start', 'test_end', 'n', 'NSE', 'KGE', 'RMSE', 'MAE', 'bias_percent', 'status'])
summary_rows = []
for tag in ('Seen', 'Unseen'):
    rows = [r for r in all_metrics if r['seen_unseen'] == tag]
    scored = [r for r in rows if np.isfinite(r['NSE'])]
    summary_rows.append(dict(seen_unseen=tag, total_basins=len(rows), scored_basins=len(scored), median_NSE=float(np.median([r['NSE'] for r in scored])), median_KGE=float(np.nanmedian([r['KGE'] for r in scored])), median_bias_percent=float(np.nanmedian([r['bias_percent'] for r in scored]))))
save_csv(CSV_PATH.parent / 'model_performance_summary.csv', summary_rows)
atomic(OUT / 'summary.json', dict(model='Daymet + PCA20', best_epoch=ck['epoch'], test_start='2005-01-01', test_end='2009-12-31', groups=summary_rows))
atomic(OUT / 'status.json', dict(phase='complete', basins=1426, seen=1326, unseen=100))
print('Both test groups complete. Basin-level metrics exported beside this notebook.')


TEST unseen100 100 / 100


TEST donor_temporal 100 / 1326


TEST donor_temporal 200 / 1326


TEST donor_temporal 300 / 1326


TEST donor_temporal 400 / 1326


TEST donor_temporal 500 / 1326


... (6 outputs left out)

TEST donor_temporal 1200 / 1326


TEST donor_temporal 1300 / 1326


TEST donor_temporal 1326 / 1326


Both test groups complete. Basin-level metrics exported beside this notebook.


## 5. Check performance and export
One row per basin, including all 1,426 IDs and Seen/Unseen. No-observation basins remain present with missing metrics. Results are saved in this experiment directory.

In [7]:
from pathlib import Path
import csv, json, html
from IPython.display import display, HTML
B = Path.cwd()
OUT = B / 'results_seed46'
summary_rows = list(csv.DictReader((B / 'model_performance_summary.csv').open()))
performance = list(csv.DictReader((B / 'basin_model_performance.csv').open()))

def table(rows):
    keys = list(rows[0])
    return '<table><tr>' + ''.join(('<th>' + html.escape(k) + '</th>' for k in keys)) + '</tr>' + ''.join(('<tr>' + ''.join(('<td>' + html.escape(str(r[k])) + '</td>' for k in keys)) + '</tr>' for r in rows)) + '</table>'
display(HTML('<h3>Test performance: 2005–2009</h3>' + table(summary_rows)))
display(HTML('<h3>First 20 basin results</h3>' + table(performance[:20])))
history = json.loads((OUT / 'history.json').read_text())
assert len(history) == 20 and len(performance) == 1426
best = json.loads((OUT / 'summary.json').read_text())['best_epoch']
(B / 'completion.json').write_text(json.dumps(dict(status='complete', epochs=len(history), best_epoch=best, basins=1426, seen=1326, unseen=100), indent=2))
print('Completed all 20 epochs and both test groups. All 1426 basin rows exported; See basin_model_performance.csv.')


seen_unseen,total_basins,scored_basins,median_NSE,median_KGE,median_bias_percent
Seen,1326,1317,0.6882783731314048,0.6645483281842189,-3.5561488313365697
Unseen,100,100,0.5975163229611928,0.5433036097784991,-5.1425900540284974


basin_id,seen_unseen,catchment_split,model,test_start,test_end,n,NSE,KGE,RMSE,MAE,bias_percent,status
CAN_01AD003,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8066377055355889,0.718695786188942,1.2374374294414132,0.608501992320371,-11.158023259108504,scored
CAN_01AF007,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8662858966571021,0.8242545935592163,1.2538828716017676,0.6779751317752751,-13.139994976936102,scored
CAN_01AF009,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8141238819808996,0.7800302914388101,1.3334443402563771,0.7548559690228804,-17.232717229771914,scored
CAN_01AJ003,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8568841327818486,0.8906247995622394,1.141135177987776,0.5408632287402765,-0.7196795813844469,scored
CAN_01AJ004,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8115926718449062,0.8462220737549433,1.2588053025990646,0.6076673277509066,-8.873795107976614,scored
CAN_01AJ010,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.7825818932085185,0.8470088214578846,1.455910019673226,0.7560642231982867,-5.100131182436933,scored
CAN_01AK001,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.8705945770630966,0.8866960266840269,0.9227764677755639,0.5714178028991975,5.01163834052545,scored
CAN_01AK006,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.6738901498397931,0.5975688695945098,1.8528928091856103,1.1511630545971476,17.923228970193392,scored
CAN_01AK007,Seen,training_donor,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.7457882957366568,0.7403221260187466,1.7050442089574414,0.8246513848788192,-2.579302561999204,scored
CAN_01AL002,Unseen,unseen_test,Daymet + PCA20,2005-01-01,2009-12-31,1826,0.745995051017752,0.7707184009232247,1.7962561030637711,0.9399637899670008,-14.513229802714191,scored


Completed all 20 epochs and both test groups. All 1426 basin rows exported; See basin_model_performance.csv.
